In [1]:
import os
import json
import time
import random
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

from openai import OpenAI
oai = OpenAI()

llm = ChatOpenAI(model="gpt-4o-mini")


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:
WORK_DIR = Path('./260519_ft')
WORK_DIR.mkdir(exist_ok=True)

In [3]:
from dataclasses import dataclass, field

In [4]:
@dataclass
class FTState:
    train_path : Path | None = None
    val_path : Path | None = None
    train_file_id: str | None = None
    val_file_id: str | None = None
    job_id: str | None = None
    ft_model: str | None = None
    notes: list = field(default_factory=list)

In [5]:
state = FTState()

In [6]:
for msg in ['start', 'data ready', 'upload done']:
    state.notes.append(msg)

In [7]:
state

FTState(train_path=None, val_path=None, train_file_id=None, val_file_id=None, job_id=None, ft_model=None, notes=['start', 'data ready', 'upload done'])

In [8]:
# 입력 : 무례하거나 짧은 한국어
# 출력 : 같은 의미의 정중한 한국어 이메일 문장

In [9]:
SYSTEM_MSG = "당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다."

TASK_SPEC = {
    'name' : 'polite_email_rewriter',
    'input' : '무례하거나 짧은 한국어 한 줄',
    'output' : '정중한 한국어 이메일 문장 1~2 문장',
    'min_examples' : 30,
    'system' : SYSTEM_MSG
}

print(json.dumps(TASK_SPEC, ensure_ascii=False, indent=2))

{
  "name": "polite_email_rewriter",
  "input": "무례하거나 짧은 한국어 한 줄",
  "output": "정중한 한국어 이메일 문장 1~2 문장",
  "min_examples": 30,
  "system": "당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다."
}


In [10]:
seed_pairs = [
    ("회의 내일로 미뤄.",   "회의를 내일로 변경 가능할지 여쭙고 싶습니다."),
    ("문서 빨리 줘.",       "문서 전달 가능 시점을 알려주실 수 있을까요?"),
    ("이거 다시 해.",       "이 부분은 한 번 더 수정 부탁드려도 될까요?"),
    ("일정 잡아.",          "편하신 일정을 알려주시면 회의를 잡아 두겠습니다."),
    ("왜 답장 안 해.",      "혹시 이전 메일을 확인하셨을지 여쭙고 싶습니다."),
    ("이건 틀렸어.",        "이 부분은 다시 한 번 검토가 필요해 보입니다."),
]

In [11]:
def to_jsonl_line(system, user, assistant) -> str:
    return json.dumps({'messages' : [
        {'role' : 'system', 'content' : system},
        {'role' : 'user', 'content' : user},
        {'role' : 'assistant', 'content' : assistant},
    ]}, ensure_ascii=False)

In [12]:
seed_path = WORK_DIR / 'seed.jsonl'  # Path('./260509_ft/seed.jsonl')
with open(seed_path, 'w', encoding='utf-8') as f:
    for r, p in seed_pairs:
        f.write(to_jsonl_line(SYSTEM_MSG, r, p) + '\n')

In [13]:
extras = [
    ("자료 빠뜨렸어.",     "자료 한 가지가 누락된 것 같습니다. 확인 부탁드립니다."),
    ("일정 다시 봐.",      "일정을 다시 한 번 확인해 주실 수 있을까요?"),
    ("이름 잘못 적었어.",  "이름 표기를 한 번 더 확인해 주시면 감사하겠습니다."),
    ("결재 빨리 해.",      "결재 처리 가능 시점을 알려주실 수 있을까요?"),
]

with open(seed_path, 'a', encoding='utf-8') as f:
    for r, p in extras:
        f.write(to_jsonl_line(SYSTEM_MSG, r, p) + '\n')

In [14]:
import re
AUG_SYS = '당신은 학습용 데이터 페어를 생성하는 전문가입니다. 출력은 반드시 JSON 배열로'
def make_augment_prompt(pairs, n=8):
    sample = '\n'.join(f'- "{r}" -> "{p}"' for r, p in pairs[:4])
    return (f"다음은 무례한 문장 -> 정중한 이메일 문장 페어입니다.\n"
            f"같은 스타일,길이 분포로 새 페어 {n} 쌍을 한국어로 만드세요.\n"
            f"출력 형식: [{{\"rude\":\"...\",\"polite\":\"...\"}}, ...]\n\n예시:\n{sample}")

resp = llm.invoke([SystemMessage(content=AUG_SYS), 
                  HumanMessage(content=make_augment_prompt(seed_pairs, n=10))]).content
print(resp[:100])

[
    {"rude":"자료 좀 보내줘.","polite":"자료를 보내주실 수 있으신지 여쭙고 싶습니다."},
    {"rude":"이 일 언제 끝내.","polite":"


In [15]:
print(resp[:300])

[
    {"rude":"자료 좀 보내줘.","polite":"자료를 보내주실 수 있으신지 여쭙고 싶습니다."},
    {"rude":"이 일 언제 끝내.","polite":"이 일을 완료하실 일정이 언제인지 알려주시면 감사하겠습니다."},
    {"rude":"지금 당장 확인해.","polite":"현재 상황을 확인해 주실 수 있으신가요?"}, 
    {"rude":"회의 시간 정해.","polite":"회의 시간을 정해 주시면 감사하겠습니다."},
    {"rude":"이거 나한테 맡겨.","polite":"이 업무를 


In [16]:
# gpt-4o-mini -> gpt-4o-mini-260519 -> gpt-4o-mini-260519_2

In [17]:
def parse_pairs(text):
    m = re.search(r"\[.*\]", text, re.DOTALL)
    if not m:
        return []
    try:
        return [(it['rude'], it['polite']) for it in json.loads(m.group(0))]
    except Exception:
        return []

augmented = parse_pairs(resp)

In [18]:
augmented

[('자료 좀 보내줘.', '자료를 보내주실 수 있으신지 여쭙고 싶습니다.'),
 ('이 일 언제 끝내.', '이 일을 완료하실 일정이 언제인지 알려주시면 감사하겠습니다.'),
 ('지금 당장 확인해.', '현재 상황을 확인해 주실 수 있으신가요?'),
 ('회의 시간 정해.', '회의 시간을 정해 주시면 감사하겠습니다.'),
 ('이거 나한테 맡겨.', '이 업무를 제가 맡아도 괜찮을지 여쭙고 싶습니다.'),
 ('보고서 이상해.', '보고서에 대해 몇 가지 수정이 필요할 것 같아, 검토해 주실 수 있으신가요?'),
 ('이번 주 꼭 해.', '이번 주 내로 진행해 주실 수 있도록 부탁드립니다.'),
 ('그거 내일까지 해.', '그 작업을 내일까지 완료해 주실 수 있을까요?'),
 ('자세히 말해줬으면 해.', '이 부분에 대해 좀 더 자세히 설명해 주실 수 있으신가요?'),
 ('답장 빨리 해.', '답장 주시면 매우 감사하겠습니다.')]

In [19]:
# -> llm -> evaluate -> 좋은애들만 남겨라

In [20]:
import tiktoken

In [21]:
# 구조, 길이(빈 응답), 중복 

In [22]:
enc = tiktoken.get_encoding('cl100k_base')

In [23]:
MAX_TOKENS_PER_SAMPLE = 4096

In [24]:
def check_sample(row) -> list[str]:
    errs = []
    msgs = row.get('messages', [])
    roles = {m.get('role') for m in msgs}
    if not {'system', 'user', 'assistant'} <= roles:
        errs.append('필수 role 누락')
    if not any(m.get('role') == 'assistant' and m.get('content') for m in msgs):
        errs.append('assistant 응답 누락')
    total = sum(len(enc.encode(m.get('content', ''))) for m in msgs)
    if total > MAX_TOKENS_PER_SAMPLE:
        errs.append('토큰 개수 초과')
    return errs

In [25]:
def validate(path):
    seen, bad, rows = set(), [], []
    for i, line in enumerate(open(path, encoding='utf-8')):
        try:
            row = json.loads(line)
        except:
            bad.append((i, "json 파싱 실패"))
            continue
        errs = check_sample(row)
        user_msg = next((m['content'] for m in row.get('messages', []) if m['role'] == 'user'), "")
        if user_msg in seen:
            errs.append('user 중복')
        seen.add(user_msg)
        (bad if errs else rows).append((i, ", ".join(errs)) if errs else row)
    return rows, bad

In [26]:
seed_path

PosixPath('260519_ft/seed.jsonl')

In [27]:
# 페어 리스트 -> jsonl 빌드
def pairs_to_jsonl(pairs : list[tuple[str, str]], system : str, out_path : Path) -> int:
    n=0
    with open(out_path, 'w', encoding='utf-8') as f:
        for r, p in pairs:
            f.write(to_jsonl_line(system, r, p) + "\n")
            n +=1
    return n

In [28]:
augmented

[('자료 좀 보내줘.', '자료를 보내주실 수 있으신지 여쭙고 싶습니다.'),
 ('이 일 언제 끝내.', '이 일을 완료하실 일정이 언제인지 알려주시면 감사하겠습니다.'),
 ('지금 당장 확인해.', '현재 상황을 확인해 주실 수 있으신가요?'),
 ('회의 시간 정해.', '회의 시간을 정해 주시면 감사하겠습니다.'),
 ('이거 나한테 맡겨.', '이 업무를 제가 맡아도 괜찮을지 여쭙고 싶습니다.'),
 ('보고서 이상해.', '보고서에 대해 몇 가지 수정이 필요할 것 같아, 검토해 주실 수 있으신가요?'),
 ('이번 주 꼭 해.', '이번 주 내로 진행해 주실 수 있도록 부탁드립니다.'),
 ('그거 내일까지 해.', '그 작업을 내일까지 완료해 주실 수 있을까요?'),
 ('자세히 말해줬으면 해.', '이 부분에 대해 좀 더 자세히 설명해 주실 수 있으신가요?'),
 ('답장 빨리 해.', '답장 주시면 매우 감사하겠습니다.')]

In [29]:
aug_path = WORK_DIR / 'aug.jsonl'
n = pairs_to_jsonl(augmented, SYSTEM_MSG, aug_path)
n

10

In [30]:
combined = WORK_DIR / 'combined.jsonl'
with open(combined, 'w', encoding='utf-8') as f:
    for src in [seed_path, aug_path]:
        if src.exists():
            f.write(open(src).read())

ok, bad = validate(combined)

In [31]:
len(ok), len(bad)

(20, 0)

In [32]:
bad

[]

In [33]:
def filter_clean(in_path:Path, out_path:Path) -> dict:  # jsonl 파일을 받아서 -> validate Ok, Bad -> ok만 out_path에 저장
    ok_rows, bad_rows = validate(in_path)
    with open(out_path, 'w', encoding='utf-8') as f:
        for row in ok_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    total = len(ok_rows) + len(bad_rows)
    pct = (len(ok_rows) / total * 100) if total else 0.0
    return {'ok' : len(ok_rows), 'bad' : len(bad_rows), 'pass_rate' : pct}

In [34]:
clean_path = WORK_DIR / 'clean.jsonl'
res = filter_clean(combined, clean_path)

In [35]:
res

{'ok': 20, 'bad': 0, 'pass_rate': 100.0}

In [36]:
# train val test

In [37]:
def split_jsonl(in_path, train_path, val_path, val_ratio = 0.2, seed = 42):
    rows = [json.loads(l) for l in open(in_path, encoding='utf-8')]
    random.Random(seed).shuffle(rows)
    n_val = max(1, int(len(rows) * val_ratio))
    val, train = rows[:n_val], rows[n_val:]
    for path, data in [(train_path, train), (val_path, val)]:
        with open(path, 'w', encoding='utf-8') as f:
            for row in data:
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
    return len(train), len(val)

In [38]:
state.train_path = WORK_DIR / 'train.jsonl'
state.val_path = WORK_DIR / 'val.jsonl'

n_tr, n_vl = split_jsonl(clean_path, state.train_path, state.val_path)

In [39]:
n_tr, n_vl

(16, 4)

In [40]:
# 100
# 80%      10%     10%         -> 모델성능
# 한국어    영어    스페인어

In [41]:
# k-fold validation
# 80% train 20% validation
# 100만개
# 20_ 20만_ _ _ _
# 1 2 3 4 5 -> 1,2,3,4 -> 5
#              1,2,3,5 -> 4
#              1,2,4,5 -> 3


In [42]:
def kfold_indices(n, k, seed=42) -> list[tuple[list[int], list[int]]]:
    if k>n:
        raise ValueError(f"k({k}) > n({n})")
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    folds = []
    for i in range(k):
        val = idx[i::k]
        train = [x for x in idx if x not in set(val)]
        folds.append((train, val))
    return folds

In [43]:
folds = kfold_indices(10, 5)
folds

[([3, 2, 8, 5, 9, 4, 0, 1], [7, 6]),
 ([7, 2, 8, 5, 6, 4, 0, 1], [3, 9]),
 ([7, 3, 8, 5, 6, 9, 0, 1], [2, 4]),
 ([7, 3, 2, 5, 6, 9, 4, 1], [8, 0]),
 ([7, 3, 2, 8, 6, 9, 4, 0], [5, 1])]

In [44]:
# oai.files.create(file=파일경로, purpose='fine-tune')
def upload_jsonl(path):
    with open(path, 'rb') as f:
        result = oai.files.create(file=f, purpose='fine-tune')
    return result.id

In [45]:
state

FTState(train_path=PosixPath('260519_ft/train.jsonl'), val_path=PosixPath('260519_ft/val.jsonl'), train_file_id=None, val_file_id=None, job_id=None, ft_model=None, notes=['start', 'data ready', 'upload done'])

In [46]:
state.train_file_id = upload_jsonl(state.train_path)
state.val_file_id = upload_jsonl(state.val_path)

In [47]:
state.train_file_id, state.val_file_id

('file-6xAvKmqZBcyL7KVChQNSFg', 'file-72ajuGrgLxGy4MGCZpBZHH')

In [48]:
oai.files.list().data[:5]

[FileObject(id='file-72ajuGrgLxGy4MGCZpBZHH', bytes=1207, created_at=1779277527, filename='val.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None),
 FileObject(id='file-6xAvKmqZBcyL7KVChQNSFg', bytes=4968, created_at=1779277527, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None),
 FileObject(id='file-93zmKCVVPTjTZyHK4TmcPi', bytes=1203, created_at=1779276229, filename='val.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None),
 FileObject(id='file-8HWhKnsuxay6tLJ2sDsWfX', bytes=4801, created_at=1779276229, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None),
 FileObject(id='file-JR4z8UQ3heJuonABiRikmZ', bytes=2361, created_at=1779177770, filename='mini2_val.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)]

In [49]:
def list_ft_files(client, purpose = 'fine-tune', limit=5):
    items = client.files.list().data
    matched = [f for f in items if getattr(F, 'purpose', '') == purpose]
    matched.sort(key = lambda f: getattr(f, "created_at", 0), reverse=True)
    return [{'id' : f.id, "filename":f.filename, 'bytes' : getattr(f, 'bytes', 0)} for f in matched[:limit]]

In [50]:
# !curl https://api.openai.com/v1/models -H 'Authorization: Bearer $OPENAI_API_KEY'

In [51]:
BASE_MODEL = 'gpt-4o-mini-2024-07-18'

In [52]:
state

FTState(train_path=PosixPath('260519_ft/train.jsonl'), val_path=PosixPath('260519_ft/val.jsonl'), train_file_id='file-6xAvKmqZBcyL7KVChQNSFg', val_file_id='file-72ajuGrgLxGy4MGCZpBZHH', job_id=None, ft_model=None, notes=['start', 'data ready', 'upload done'])

In [75]:
job = oai.fine_tuning.jobs.create(
    training_file = state.train_file_id,
    validation_file = state.val_file_id,
    model = BASE_MODEL,
    suffix = "polite-email",
    hyperparameters = {'n_epochs' : 'auto'}
)

state.job_id = job.id

PermissionDeniedError: Error code: 403 - {'error': {'message': 'OpenAI is winding down the fine-tuning platform and your organization is no longer able to create new fine-tuning training jobs. Learn more https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning', 'type': 'invalid_request_error', 'param': None, 'code': 'training_not_available'}}

In [68]:
oai.models.list().data

[Model(id='text-embedding-ada-002', created=1671217299, object='model', owned_by='openai-internal'),
 Model(id='whisper-1', created=1677532384, object='model', owned_by='openai-internal'),
 Model(id='gpt-3.5-turbo', created=1677610602, object='model', owned_by='openai'),
 Model(id='tts-1', created=1681940951, object='model', owned_by='openai-internal'),
 Model(id='gpt-3.5-turbo-16k', created=1683758102, object='model', owned_by='openai-internal'),
 Model(id='gpt-4-0613', created=1686588896, object='model', owned_by='openai'),
 Model(id='gpt-4', created=1687882411, object='model', owned_by='openai'),
 Model(id='davinci-002', created=1692634301, object='model', owned_by='system'),
 Model(id='babbage-002', created=1692634615, object='model', owned_by='system'),
 Model(id='gpt-3.5-turbo-instruct', created=1692901427, object='model', owned_by='system'),
 Model(id='gpt-3.5-turbo-instruct-0914', created=1694122472, object='model', owned_by='system'),
 Model(id='gpt-3.5-turbo-1106', created=16

In [ ]:
# 1~ 100k : 모델이 한번 보고 학습(모델의 파라미터를 업데이트) -> 1 epoch

# n_epoch : 100 -> 
# 0.01, 0.001 learning_rate

In [65]:
j = oai.fine_tuning.jobs.retrieve('ftjob-JerImXCzch54WKIAwBjcOjlg')

In [67]:
j.id, j.status, j.model, j.trained_tokens

('ftjob-JerImXCzch54WKIAwBjcOjlg', 'succeeded', 'gpt-4o-mini-2024-07-18', 5832)

In [63]:
events = oai.fine_tuning.jobs.list_events(
    fine_tuning_job_id = 'ftjob-JerImXCzch54WKIAwBjcOjlg', limit = 50)

for e in events.data:
    print(e.created_at, e.message)

1779160068 The job has successfully completed
1779160065 Usage policy evaluations completed, model is now enabled for sampling
1779160065 Moderation checks for snapshot ft:gpt-4o-mini-2024-07-18:ai-data-short-term:polite-email:Dh4tr9SQ passed.
1779159312 Evaluating model against our usage policies
1779159312 New fine-tuned model created
1779159312 Checkpoint created at step 80
1779159312 Checkpoint created at step 64
1779159257 Step 96/96: training loss=0.00, validation loss=0.70, full validation loss=1.00
1779159253 Step 95/96: training loss=0.00, validation loss=0.92
1779159249 Step 94/96: training loss=0.00, validation loss=1.82
1779159246 Step 93/96: training loss=0.00, validation loss=0.34
1779159243 Step 92/96: training loss=0.03, validation loss=0.72
1779159239 Step 91/96: training loss=0.00, validation loss=1.81
1779159239 Step 90/96: training loss=0.00, validation loss=0.33
1779159236 Step 89/96: training loss=0.01, validation loss=0.90
1779159232 Step 88/96: training loss=0.0

In [60]:
def build_job_payload(state, base_model, suffix, n_epochs='auto') -> dict:
    if not state.train_file_id:
        raise ValueError('train 파일 미업로드')
    if len(suffix) > 20:
        raise ValueError('suffix length')
    payload = {
        'model' : base_model,
        'training_file' : state.train_file_id,
        'suffix' : suffix,
        'hyperparameters' : {'n_epochs' : n_epochs}
    }
    if state.val_file_id:
        payload['validation_file'] = state.val_file_id
    return payload

In [69]:
# Model(id='ft:gpt-4o-mini-2024-07-18:ai-data-short-term:polite-email:Dh4tr9SQ', created=1779159312, object='model', owned_by='ai-data-short-term')

In [70]:
state.ft_model = 'ft:gpt-4o-mini-2024-07-18:ai-data-short-term:polite-email:Dh4tr9SQ'

In [71]:
llm_ft = ChatOpenAI(model = state.ft_model)

In [72]:
SYSTEM_MSG

'당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다.'

In [73]:
llm_ft.invoke([SystemMessage(content = SYSTEM_MSG), HumanMessage(content = '회의 내일로 미뤄')])

AIMessage(content='회의를 내일로 변경 가능할지 여쭙고 싶습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 45, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'ft:gpt-4o-mini-2024-07-18:ai-data-short-term:polite-email:Dh4tr9SQ', 'system_fingerprint': 'fp_bfaadcaaa2', 'id': 'chatcmpl-Dha5ZHT07TTCpqfQbJ20RwEpIQYkI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e454d-ffd9-7190-a1ca-b342a390e2be-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 17, 'total_tokens': 62, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [74]:
llm.invoke([SystemMessage(content = SYSTEM_MSG), HumanMessage(content = '회의 내일로 미뤄')])

AIMessage(content='회의 날짜를 내일로 변경해 주시면 감사하겠습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 45, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9ed8b4905a', 'id': 'chatcmpl-Dha6BrCJ3hhB2v58fYPIEt74HyuPu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e454e-9828-7fe2-8c59-d42beaecd425-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 14, 'total_tokens': 59, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
class PoliteRewriter:
    def __init__(self, model_name):
        self.model_name = model_name
        self._cache : dict[str, str] = {}
        self.hits = 0
        self.misses = 0
    
    def rewrite(self, text : str) -> str:
        if text in self._cache:
            self.hits += 1
            return self._cache[text]
        self.misses +=1
        llm = ChatOpenAI(model = self.model_name)
        out = llm.invoke([SystemMessage, HumanMessage])
        return out
    
    def batch(self, texts : list[str]) -> list[str]:
        return [self.rewrite(t) for t in texts]
    
#     회의 내일로 미뤄 -> cache 
#     회의 내일로 미뤄 -> cache hit

In [ ]:
huggingface